In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import tiktoken
import json
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from groq import Groq
import os
from dotenv import load_dotenv
import random

load_dotenv()  # Load environment variables from a .env file if present
client = Groq(api_key=os.getenv("groq_api_key"))

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import time
import pandas as pd
from groq import Groq
from tqdm import tqdm
from datetime import datetime

class GroqBatchProcessor:
    """Process batches of  problems with Groq OSS-120B"""
    
    def __init__(
        self, 
        target_model="openai/gpt-oss-120b",
        system_prompt="",
        max_retries=3,
        retry_delay=2
    ):
        self.client = Groq(api_key=os.getenv("GROQ_API_KEY"))
        self.target_model = target_model
        self.system_prompt = system_prompt
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        
        print(f"System prompt set to: {self.system_prompt}")
        
    def process_single(self, problem_text, reasoning_effort="medium", max_tokens=2048):
        """Process a single problem with retries"""
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"{problem_text}"}
        ]
        
        for attempt in range(self.max_retries):
            try:
                chat_completion = self.client.chat.completions.create(
                    model=self.target_model,
                    messages=messages,
                    temperature=0.9,
                    max_completion_tokens=max_tokens,
                    top_p=1,
                    reasoning_effort=reasoning_effort,
                    stream=False,
                    stop=None
                )
                
                msg = chat_completion.choices[0].message
                
                # Extract reasoning and response
                response = msg.content.strip() if msg.content else ""
                reasoning = msg.reasoning.strip() if hasattr(msg, 'reasoning') and msg.reasoning else ""
                
                # Get token counts
                usage = chat_completion.usage
                input_tokens = usage.prompt_tokens if hasattr(usage, 'prompt_tokens') else 0
                output_tokens = usage.completion_tokens if hasattr(usage, 'completion_tokens') else 0
                
                return {
                    'reasoning': reasoning,  # OSS-120B reasoning trace
                    'response': response,    # Final solution/answer
                    'input_tokens': input_tokens,
                    'output_tokens': output_tokens,
                    'total_tokens': input_tokens + output_tokens,
                    'success': True,
                    'error': None
                }
                
            except Exception as e:
                if attempt < self.max_retries - 1:
                    print(f"Attempt {attempt + 1} failed: {e}. Retrying in {self.retry_delay}s...")
                    time.sleep(self.retry_delay)
                else:
                    print(f"Failed after {self.max_retries} attempts: {e}")
                    return {
                        'reasoning': None,
                        'response': None,
                        'input_tokens': 0,
                        'output_tokens': 0,
                        'total_tokens': 0,
                        'success': False,
                        'error': str(e)
                    }
        
    def process_batch(
        self, 
        df, 
        input_column='input',
        uid_column='uid',
        reasoning_effort="medium",
        max_tokens=2048,
        batch_size=10,
        save_interval=100,
        output_file='oss120b_results.parquet'
    ):
        """Process batch and save reasoning + response"""
        
        results = []
        result_master=[]
        start_time = time.time()
        
        print(f"Processing {len(df)} problems with {self.target_model}")
        print(f"Reasoning effort: {reasoning_effort}, Max tokens: {max_tokens}\n")
        

        
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
            problem = row[input_column]
            
            # Process single problem
            result = self.process_single(
                problem_text=problem,
                reasoning_effort=reasoning_effort,
                max_tokens=max_tokens
            )
            
            # Add original data + metadata
            result.update({
                'uid': row[uid_column],
                'input': problem,  # Original problem
            })
            
            results.append(result)
            
            
            # Rate limiting
            if (idx + 1) % batch_size == 0:
                #Add batch to master
                result_master.extend(results)
                #Clear results for next batch
                results = []
                
                time.sleep(1)
            
            # Save checkpoint
            if (idx + 1) % save_interval == 0:
                self._save_checkpoint(result_master, output_file, idx + 1)
        
        
        # Add any leftover batch results
        if len(results) > 0:
            result_master.extend(results)

        # Final save with ALL columns
        df_results = pd.DataFrame(result_master)
        df_results.to_parquet(output_file, index=False)
        
        # Print summary
        elapsed = time.time() - start_time
        self._print_summary(df_results, elapsed)
        
        return df_results
    
    def _save_checkpoint(self, results, output_file, count):
        """Save intermediate results"""
        checkpoint_file = output_file.replace('.parquet', f'_checkpoint_{count}.parquet')
        df_temp = pd.DataFrame(results)
        df_temp.to_parquet(checkpoint_file, index=False)
        print(f"\n✓ Checkpoint saved: {checkpoint_file} ({len(results)} records)")
    
    def _print_summary(self, df_results, elapsed_time):
        """Print processing summary"""
        total = len(df_results)
        successful = df_results['success'].sum()
        failed = total - successful
        
        # Check reasoning availability
        has_reasoning = df_results[df_results['success'] == True]['reasoning'].notna().sum()
        
        avg_input_tokens = df_results[df_results['success'] == True]['input_tokens'].mean()
        avg_output_tokens = df_results[df_results['success'] == True]['output_tokens'].mean()
        total_tokens = df_results['total_tokens'].sum()
        
        print("\n" + "="*60)
        print("PROCESSING SUMMARY")
        print("="*60)
        print(f"Total processed: {total}")
        print(f"Successful: {successful} ({successful/total*100:.1f}%)")
        print(f"Failed: {failed} ({failed/total*100:.1f}%)")
        print(f"Has reasoning: {has_reasoning}")
        print(f"\nToken Usage:")
        print(f"  Avg input tokens: {avg_input_tokens:.0f}")
        print(f"  Avg output tokens: {avg_output_tokens:.0f}")
        print(f"  Total tokens used: {total_tokens:,}")
        print(f"\nTime: {elapsed_time/60:.1f} minutes")
        print(f"Rate: {total/(elapsed_time/60):.1f} problems/min")
        print("="*60)

In [3]:
import re

def extract_answer(text):
    """Extract answer from multiple common formats"""
    if text is None or not isinstance(text, str):
        return None
    
    # Pattern 1: \boxed{answer}
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    if match:
        return match.group(1).strip()
    
    # Pattern 2: **Answer:** followed by content
    match = re.search(r'\*\*Answer:\*\*\s*(.+?)(?:\n|\.|$)', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # Pattern 3: "Answer: X" or "The answer is X"
    match = re.search(r'(?:answer is|answer:)\s*\*?\*?(.+?)(?:\.|$)', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # Pattern 4: Last sentence with numerical result (for math)
    # Look for patterns like "Albert eats 48 pieces" or "Total = $5,250"
    sentences = text.split('.')
    for sentence in reversed(sentences):
        # Find numbers with units or currency
        match = re.search(r'(\$?[\d,]+(?:\.\d+)?(?:\s*(?:pieces|slices|dollars|items|units|people))?)', sentence)
        if match:
            return match.group(1).strip()
    
    return None



### 1. Getting the distillation reasoning and response from OSS120B model for MATH

    - reasoning = medium
    - max tokens = 1900 (output)
    - temperature = 0.9

- We will generate reasoning and response for all questions.
- We will not add any complex prompts to it. Prompt is simple : "You are an expert in math. Provide clear, concise solutions to these math problems within the token limit."
- Some answers might be wrong, some answers might be truncated. We flag these at the end

In [3]:
##Load math data
math_data = pd.read_parquet('../data/raw-data/math_base_dataset.parquet')

In [5]:
math_data.head()

,input,source_answer,split,source,domain,ground_truth,problem_type,question_type,uid
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...,train,gsm8k,math,72,,,math1
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...,train,gsm8k,math,10,,,math2
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<...",train,gsm8k,math,5,,,math3
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....,train,gsm8k,math,42,,,math4
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,train,gsm8k,math,624,,,math5


In [4]:
##Select reasoning needed only for train samples
print(f"Total Samples {len(math_data)}")
train_math_df = math_data[math_data['split'] == 'train'].copy()
print(f"Train Samples {len(train_math_df)}")

Total Samples 14822
Train Samples 13203


In [22]:
samp = train_math_df.sample(34).reset_index(drop=True)

In [41]:
# Initialize
processor = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt="You are an expert in math. Provide clear, concise solutions to these math problems within the token limit.",
    
)

# Process your filtered dataset
results_df = processor.process_batch(
    df=train_math_df.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1930,
    batch_size=100,
    save_interval=500,  # Save every 500 records
    output_file='../data/math-distillation/checkpoints/math_distillation_with_reasoning.parquet'
)
    
# Verify columns
print("\nColumns in results:")
print(results_df.columns.tolist())

System prompt set to: You are an expert in math. Provide clear, concise solutions to these math problems within the token limit.
Processing 13203 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1930



Processing:   4%|▍         | 500/13203 [06:41<3:05:18,  1.14it/s]


✓ Checkpoint saved: ../data/math-distillation/checkpoints/math_distillation_with_reasoning_checkpoint_500.parquet (500 records)


Processing:   8%|▊         | 1000/13203 [12:53<4:06:37,  1.21s/it]


✓ Checkpoint saved: ../data/math-distillation/checkpoints/math_distillation_with_reasoning_checkpoint_1000.parquet (1000 records)


...


✓ Checkpoint saved: ../data/math-distillation/checkpoints/math_distillation_with_reasoning_checkpoint_12500.parquet (12500 records)


Processing:  98%|█████████▊| 13000/13203 [6:15:54<03:23,  1.00s/it]


✓ Checkpoint saved: ../data/math-distillation/checkpoints/math_distillation_with_reasoning_checkpoint_13000.parquet (13000 records)


Processing: 100%|██████████| 13203/13203 [6:17:41<00:00,  1.72s/it]



PROCESSING SUMMARY
Total processed: 13203
Successful: 13203 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 13203

Token Usage:
  Avg input tokens: 163
  Avg output tokens: 739
  Total tokens used: 11,920,763

Time: 377.7 minutes
Rate: 35.0 problems/min

Columns in results:
['reasoning', 'response', 'input_tokens', 'output_tokens', 'total_tokens', 'success', 'error', 'uid', 'input']


In [44]:
math_final = pd.merge(math_data,results_df.drop(columns=['input']), on='uid', how='left')

In [8]:
math_final['model_answer'] = math_final['response'].apply(extract_answer)
math_final['model_answer_available'] = math_final['model_answer'].notna()
math_final['validated_correct'] = math_final['model_answer'] == math_final['ground_truth'].astype(str)

In [30]:
##Save the math base dataset
math_final.to_parquet('../data/math-distillation/math_distillation_dataset.parquet')
math_final.to_csv('../data/math-distillation/math_distillation_dataset.csv', index=False)

In [11]:
print(math_final.groupby(['model_answer_available', 'source','split'])['validated_correct'].size())

model_answer_available  source            split
False                   ChilleD/SVAMP     test      300
                        Math/AIME 2025    train      19
                        OpenR1-Math-220k  train    1297
                        gsm8k             test     1319
True                    ChilleD/SVAMP     train     700
                        Math/AIME 2025    train      11
                        OpenR1-Math-220k  train    3703
                        gsm8k             train    7473
Name: validated_correct, dtype: int64


In [5]:
math_final = pd.read_parquet('../data/math-distillation/math_distillation_dataset.parquet')

In [6]:
math_final[math_final['model_answer_available']==False].head(10)

,input,source_answer,split,source,domain,ground_truth,problem_type,question_type,uid,reasoning,response,input_tokens,output_tokens,total_tokens,success,error,model_answer,model_answer_available,validated_correct
6,Albert is wondering how much pizza he can eat ...,He eats 32 from the largest pizzas because 2 x...,train,gsm8k,math,48,,,math7,We need to compute total slices: 2 large *16 =...,Albert has \n\n- \(2\) large pizzas × \(16\) ...,151.0,127.0,278.0,True,None,None,False,False
26,Jack is stranded on a desert island. He wants ...,First find how many liters of the seawater are...,train,gsm8k,math,400,,,math27,We need to compute salt amount: 2 liters = 200...,The bucket holds \n\n\[\n2\ \text{L}=2000\ \t...,149.0,176.0,325.0,True,None,None,False,False
31,Noah is a painter. He paints pictures and sell...,Noah earned $60/large painting x 8 large paint...,train,gsm8k,math,1200,,,math32,We need to compute last month's revenue: 8 lar...,Last month’s revenue \n\n- Large paintings: \...,157.0,157.0,314.0,True,None,None,False,False
34,Mr. Sam shared a certain amount of money betwe...,Tony got twice $1750 which is 2*$1750 = $<<2*1...,train,gsm8k,math,5250,,,math35,"The user asks: ""Mr. Sam shared a certain amoun...","Ken received \$1,750. \nTony received twice a...",136.0,195.0,331.0,True,None,None,False,False
35,Mr. Sanchez found out that 40% of his Grade 5 ...,"Since 40% of his students got below B, 100% - ...",train,gsm8k,math,36,,,math36,The problem: 40% got below B. So 60% got B and...,**Solution**\n\n- Total number of Grade 5 stud...,142.0,163.0,305.0,True,None,None,False,False
37,Five friends eat at a fast-food chain and orde...,The cost of 5 pieces of hamburger is $3 x 5 = ...,train,gsm8k,math,5,,,math38,We need to calculate total cost: 5*3 = 15; 4*1...,**Step‑by‑step calculation**\n\n| Item | Quant...,173.0,322.0,495.0,True,None,None,False,False
41,Rachel and Sara want to attend a beauty and mo...,The cost Rachel should budget for her pair of ...,train,gsm8k,math,500,,,math42,We need to compute Sara total: 50+200 = 250. R...,Sara’s total spending \n\[\n50\text{ (shoes)}...,162.0,112.0,274.0,True,None,None,False,False
44,James is a first-year student at a University ...,Accommodation is 15% * $1000=$<<15*.01*1000=15...,train,gsm8k,math,300,,,math45,We need compute remaining percentage: total pe...,"James allocates his $1,000 budget as follows:\...",158.0,183.0,341.0,True,None,None,False,False
47,"The file, 90 megabytes in size, downloads at t...",The first 60 megabytes take 60/5=<<60/5=12>>12...,train,gsm8k,math,15,,,math48,We need total time. First 60 MB at 5 MB/s -> t...,The download proceeds in two phases:\n\n1. **F...,147.0,214.0,361.0,True,None,None,False,False
50,Gerald spends $100 a month on baseball supplie...,He needs to save up $400 because 4 x 100 = <<4...,train,gsm8k,math,5,,,math51,We need to parse the problem. Gerald spends $1...,Gerald’s baseball supplies cost \n\n\[\n100\t...,163.0,356.0,519.0,True,None,None,False,False
